In [50]:
import os
import re
import pandas as pd
from tqdm import tqdm

# Paths
DATA_PATH = '/data/gusev/USERS/jpconnor/data/'
EMBED_PROJ_PATH = os.path.join(DATA_PATH, 'clinical_text_embedding_project/')
NEPC_PROJ_PATH = os.path.join(DATA_PATH, 'CAIA/COMPASS/')

PROC_PATH = os.path.join(EMBED_PROJ_PATH, 'batched_datasets/processed_datasets/')
TEXT_PATH = os.path.join(EMBED_PROJ_PATH, 'batched_datasets/batched_text/')

PROFILE_PATH = '/data/gusev/PROFILE/CLINICAL/'
INTAE_DATA_PATH = os.path.join(PROFILE_PATH, 'robust_VTE_pred_project_2025_03_cohort/data/')
ONCDRS_PATH = os.path.join(PROFILE_PATH, 'OncDRS/ALL_2025_03/')

SURV_PATH = os.path.join(EMBED_PROJ_PATH, 'time-to-event_analysis/')

os.listdir(NEPC_PROJ_PATH)

lab_code_mapping = pd.read_csv('/data/gusev/USERS/jpconnor/code/CAIA/COMPASS/PROFILE/OMOP_to_DFCI_lab_ids.csv')

In [51]:
# DFCI_MRN, DATE, Test DESCR, Numeric_result, text_result, unit

In [73]:
health_df = pd.read_csv(os.path.join(NEPC_PROJ_PATH, 'prostate_health_history_data.csv'))
labs_df = pd.read_csv(os.path.join(NEPC_PROJ_PATH, 'prostate_labs_data.csv')) 

vital_signs_df = health_df.loc[health_df['CODE_TYPE'] == 'Vital Signs', 
                               ['DFCI_MRN', 'START_DT', 'HEALTH_HISTORY_TYPE', 'RESULTS', 'UNITS_CD']]

In [74]:
labs_df_col_sub = labs_df[['DFCI_MRN', 'SPECIMEN_COLLECT_DT', 'TEST_TYPE_CD', 'TEST_TYPE_DESCR', 'NUMERIC_RESULT', 'RESULT_UOM_NM']]

def generate_new_test_names(code, descr):
    if str(code).lower() == 'nan':
        return descr
    elif code == descr:
        return code
    else:
        return f'{code} ({descr})'


labs_df_col_sub['TEST_NAME'] = labs_df_col_sub.apply(lambda row : generate_new_test_names(row['TEST_TYPE_CD'], row['TEST_TYPE_DESCR']), axis=1)
labs_df_col_sub = labs_df_col_sub.drop(columns=['TEST_TYPE_CD', 'TEST_TYPE_DESCR'])

vital_signs_df = (vital_signs_df
                  .rename(columns={'START_DT' : 'DATE', 'RESULTS' : 'LAB_VALUE', 
                                   'HEALTH_HISTORY_TYPE' : 'LAB_NAME', 'UNITS_CD' : 'LAB_UNIT'})
                  [['DFCI_MRN', 'DATE', 'LAB_NAME', 'LAB_UNIT', 'LAB_VALUE']])

labs_df_col_sub = (labs_df_col_sub
                   .rename(columns={'SPECIMEN_COLLECT_DT' : 'DATE', 'NUMERIC_RESULT' : 'LAB_VALUE', 
                                    'RESULT_UOM_NM' : 'LAB_UNIT', 'TEST_NAME' : 'LAB_NAME'})
                   [['DFCI_MRN', 'DATE', 'LAB_NAME', 'LAB_UNIT', 'LAB_VALUE']])

/tmp/ipykernel_312912/531633621.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  labs_df_col_sub['TEST_NAME'] = labs_df_col_sub.apply(lambda row : generate_new_test_names(row['TEST_TYPE_CD'], row['TEST_TYPE_DESCR']), axis=1)


In [80]:
complete_longitudinal_data = pd.concat([vital_signs_df, labs_df_col_sub])

In [82]:
code_descriptions = complete_longitudinal_data[['LAB_NAME', 'LAB_UNIT']].value_counts().reset_index()
code_descriptions.to_csv('/data/gusev/USERS/jpconnor/code/CAIA/COMPASS/PROFILE/unique_lab_ids_w_units.csv', index=False)

In [84]:
complete_longitudinal_data.to_csv(os.path.join(NEPC_PROJ_PATH, 'uncondensed_longitudinal_data.csv'), index=False)

In [85]:
!python consolidate_dfci_labs.py consolidate \
--input-csv /data/gusev/USERS/jpconnor/data/CAIA/COMPASS/uncondensed_longitudinal_data.csv \
--output-csv /data/gusev/USERS/jpconnor/data/CAIA/COMPASS/consolidated_longitudinal_data.csv

/data/gusev/USERS/jpconnor/code/CAIA/COMPASS/PROFILE/data_preprocessing/consolidate_dfci_labs.py:964: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  input_df = pd.read_csv(args.input_csv)
Wrote consolidated labs to /data/gusev/USERS/jpconnor/data/CAIA/COMPASS/consolidated_longitudinal_data.csv


In [86]:
long_df = pd.read_csv(os.path.join(NEPC_PROJ_PATH, 'consolidated_longitudinal_data.csv'))

/tmp/ipykernel_312912/234338941.py:1: DtypeWarning: Columns (4,8) have mixed types. Specify dtype option on import or set low_memory=False.
  long_df = pd.read_csv(os.path.join(NEPC_PROJ_PATH, 'consolidated_longitudinal_data.csv'))


In [87]:
long_df.head()

,DFCI_MRN,DATE,TEST_NAME,RESULT_UOM_NM,NUMERIC_RESULT,TEST_NAME_PREFIX,normalized_result_uom_nm,numeric_result_as_float,bp_component,split_from_combined_bp,measurement_concept_id,omop_measurement_name,collapsed_measurement,canonical_result_uom_nm,mapping_source,conversion_factor,numeric_result_standardized,conversion_status
0,648898,2020-02-10 14:40:00,Height,inch,67.99200,Height,inch,67.992,NaN,False,3036277.0,Body height,Body height,inch,exact,1.0,67.992,mapped_no_change
1,746809,2023-10-23 13:55:00,BMI,NaN,28.89000,BMI,NaN,28.890,NaN,False,NaN,NaN,NaN,NaN,unmapped,NaN,NaN,unmapped_test_name
2,826699,2018-10-11 09:00:00,Weight,pound,146.16500,Weight,pound,146.165,NaN,False,3025315.0,Body weight,Body weight,pound,exact,1.0,146.165,mapped_no_change
3,639190,2017-08-04 12:37:00,Systolic-Epic,millimeter of mercury,126.00000,Systolic-Epic,millimeterofmercury,126.000,NaN,False,3004249.0,Systolic blood pressure,Systolic blood pressure,mmHg,exact,1.0,126.000,mapped_unit_normalized
4,782904,2021-01-19 10:55:00,Body Surface Area (BSA),NaN,2.01000,Body Surface Area,NaN,2.010,NaN,False,NaN,NaN,NaN,NaN,unmapped,NaN,NaN,unmapped_test_name


In [90]:
long_df['collapsed_measurement'].value_counts()

collapsed_measurement
Diastolic blood pressure    175346
Systolic blood pressure     175346
Body temperature            124385
Heart rate                   85971
Body weight                  81387
Respiratory rate             63510
PSA                          58875
Total protein                55800
Alkaline phosphatase         55689
Creatinine                   55620
Total bilirubin              55608
AST                          55597
ALT                          55553
Albumin                      55434
BUN                          54548
Globulin                     53684
Body height                  53438
Potassium                    53412
Glucose                      53321
Calcium                      53310
CO2                          53146
Sodium                       53144
Chloride                     53138
Neutrophils absolute         52842
Lymphocytes absolute         51868
Monocytes absolute           51785
Eosinophils absolute         51760
Basophils absolute           5145